# COMP7015 Group Project – Topic 1: Plant Disease Classification

A classifier that sees only the border of a PlantVillage photo already predicts the class far above chance (Noyan, 2022), so the background is a likely shortcut.

| Rubric criterion (Excellent) | What the notebooks do | Notebook / Section |
|---|---|---|
| Reproducible leaf-group splits, no leakage | One training series per class with official leaf ids only; split by leaf id with a fixed seed; audit that no leaf id crosses splits | 01 / 3, 5 |
| Efficient caching / IO | Images are decoded and resized **once** and cached as a uint8 tensor | 01 / 4 |
| Justified class balance | Class-weighted loss for the CNNs, tested in an ablation | 01 / 6, 02 / 9 |
| Sound fine-tuning (freeze/unfreeze, schedulers) | Two-stage fine-tuning (train head → unfreeze), AdamW, learning-rate schedules, gradient clipping, early stopping, checkpoints; three ViT fine-tuning strategies (head only, last two blocks, full) | 02 / 8–9, 03 / 10 |
| Attention / transformer modelling | Custom CNN with **Squeeze-and-Excitation channel attention** and a pre-trained **Vision Transformer (ViT-B/16)** | 02 / 7, 03 / 10 |
| Systematic tuning and ablations | Learning-rate sweep and ablations of attention, augmentation, dropout, class weighting, scheduler and batch size; ResNet18 and ViT fine-tuning strategies; ViT augmentation ablation | 02 / 9, 03 / 10 |
| Per-class metrics, macro-F1, confusion matrix, error analysis | Mean ± standard deviation over 3 training seeds, per-class F1, confusion matrices, errors that every seed makes, misclassified images, Grad-CAM | 04 / 11, 13 |
| Compute trade-offs | Parameters, model size, inference latency; trainable parameters, training time and peak memory | 03 / 10, 04 / 12 |
| Background bias / robustness | Same test leaves with background removed, grayscale, darker, blurred and noisy; ViT retrained on grayscale and background-removed photos; the same leaf on a background of its own and of another class; Grad-CAM weight on the leaf | 05 / 14–17 |
| Countermeasure for the background shortcut | Training with random backgrounds, with a class-matched control | 05 / 18 |
| Beyond controlled photos | A healthy photo series left out of training and PlantDoc field photos | 05 / 19 |
| Deterministic, config-driven | One `CFG` dictionary, saved with every run; fixed seeds; deterministic algorithms; runs on CUDA, Apple MPS and CPU | 01 / 1 |

**Notebooks.** The project is split into six notebooks that run in order: `01_data` → `02_cnn_resnet` → `03_vit` → `04_test_evaluation` → `05_robustness` → `06_discussion`. The shared code (data protocol, models, training engine, evaluation tools) lives in the `plantdisease/` package; the notebooks show the steps and the results.

**Uncertainty.** All models are trained with three seeds (62, 63, 64) on one fixed split. Results are reported as the mean ± sample standard deviation over the seeds. Two models trained with the same seeds are compared seed by seed ("higher in k of 3 seeds"). The standard deviation reflects the randomness of training; it does not cover a different choice of test leaves.

**Runtime.** Every training run is saved under `OUT_DIR/runs/`. Re-running a notebook loads finished runs instead of retraining them, so an interrupted run can simply be resumed. A GPU (CUDA or Apple MPS) is recommended; notebook 04 reports the training time of every model.

## 1. Setup & Configuration

Paths, epochs and seeds can also be overridden with environment variables without editing the code, e.g. `EPOCHS_CNN=1 SEEDS=62` for a quick test. The configuration is saved to `OUT_DIR/config.json`. The notebooks need Python 3.10 or newer, PyTorch 2.0 or newer and torchvision 0.15 or newer (`transforms.v2`). They pick the device automatically: CUDA, then Apple MPS, then CPU.

## 3. Leaf Grouping & Label Verification

In [ ]:
# Label check (all photos, before any filtering): the class stored in the leaf map must agree with the folder label
map_classes = all_df["key"].map(
    lambda k: {e.split(":::")[0] for e in leaf_map.get(k, [])}
)
conflicts = sum(
    len(m) > 0 and c not in m
    for c, m in zip(all_df["class_name"], map_classes, strict=True)
)
print("Images whose leaf-map class disagrees with the folder label:", conflicts)

# Image check: every file must open; record (size, mode)
image_info = Counter()
for p in df["path"]:
    with Image.open(p) as img:
        image_info[(img.size, img.mode)] += 1
print("Image (size, mode) counts:", dict(image_info))

## 4. Image Cache & Background Analysis

**Background.** We measure the brightness of a 12-pixel border around every image, to see whether the photo background differs between classes. Every class comes from its own photo series, so it may, and a model could learn the background instead of the disease.

In [ ]:
# Background brightness: mean gray level of a 12-pixel border around each image
df["border_brightness"] = chunked(border_brightness, IMAGES).numpy()

fig, axes = plt.subplots(
    1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1.3, 1]}
)
for c, name in enumerate(SHORT_NAMES):
    axes[0].hist(
        df.loc[df.label == c, "border_brightness"],
        bins=50,
        range=(0, 255),
        histtype="step",
        linewidth=2,
        label=name,
    )
axes[0].set_xlabel("mean border brightness (0 = black)")
axes[0].set_ylabel("images")
axes[0].set_title("Background brightness per class")
axes[0].legend()
examples = [df.index[df.label == c][0] for c in range(len(CLASSES))]
grid = make_grid(IMAGES[examples], nrow=len(CLASSES), padding=2)
axes[1].imshow(grid.permute(1, 2, 0))
axes[1].set_title("One example per class: " + ", ".join(SHORT_NAMES), fontsize=9)
axes[1].axis("off")
finish("background_brightness.png")

df.groupby("class_name")["border_brightness"].agg(
    ["mean", "std", "min", "max"]
).reindex(CLASSES).round(0)